In [ ]:
#gen ai dataset
%pip install --upgrade --quiet google-genai
import sys
import time

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
from IPython.display import HTML, Markdown, display
from google import genai
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    GoogleSearch,
    HarmBlockThreshold,
    HarmCategory,
    MediaResolution,
    Part,
    Retrieval,
    SafetySetting,
    Tool,
    ToolCodeExecution,
    VertexAISearch,
)

import os
df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/train/human_1000.csv')
PROJECT_ID = "project-xxxxxxxxx"
#LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

client = genai.Client(vertexai=True, project=PROJECT_ID, location="us-central1")
#MODEL_ID = "gemini-2.5-flash"
#response = client.models.generate_content(
#model=MODEL_ID, contents="Given the invention title: 'Method for stripping aluminum from a diffusion coating', write patent claim 1."

results = []
for i, row in df.iterrows():
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Given the invention title: '{row['title']}', write patent claim 1. Use standard patent claim formatting with semicolons and no lettered or numbered lists."
        )
        results.append({
            'publication_number': row['publication_number'],
            'title': row['title'],
            'claim1': response.text,
            'label': 'ai_generated'
        })
    except Exception as e:
        print(f"Error at {i}: {e}")
        time.sleep(30)
        continue

    if i % 50 == 0:
        print(f"{i}/1000 done")
        pd.DataFrame(results).to_csv(
            '/content/drive/MyDrive/experiment1/dataset/train/ai_1000.csv',
            index=False
        )

    time.sleep(1)

pd.DataFrame(results).to_csv(
    '/content/drive/MyDrive/experiment1/dataset/train/ai_1000.csv',
    index=False
)
print("done")